## Lightweight Segmentation with a MobileNet Encoder-Decoder network

## Training

In [ ]:
# =========================
# 2) IMPORTS
# =========================
import os, time, math
import numpy as np
import pandas as pd
from PIL import Image

import cv2
import albumentations as A

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

import segmentation_models_pytorch as smp

ModuleNotFoundError: No module named 'segmentation_models_pytorch'

In [ ]:
# =========================
# 3) CONFIG – EDIT ME
# =========================
SPLITS_DIR = "Data/training_set/slz_out/det_obb/splits_yolo_obb"         # contains train.txt / val.txt / test.txt
MASKS_DIR  = "Data/training_set/gt/semantic/labels_images"               # color PNG masks
CLASS_CSV  = "Data/training_set/gt/semantic/class_dict.csv"              # 'name,r,g,b' header

# Candidate locations of the images referenced by your split IDs.
# The resolver will try these in order (and will also honor absolute/relative paths in the split files).
IMAGE_DIR_CANDIDATES = [
    "Data/training_set/slz_out/det_obb/images",
    "Data/training_set/images",
    "Data/training_set/gt/semantic/images",
    ".",  # last resort: treat paths in split files as-is
]
SPLIT_FILENAMES = {"train": "train.txt", "val": "val.txt", "test": "test.txt"}  # change if needed

# I/O + training params
OUTPUT_DIR   = "checkpoints_unet_mobilenet"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_H, IMG_W = 704, 1056          # same as your previous pipeline
BATCH_SIZE   = 3
EPOCHS       = 10
MAX_LR       = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = min(8, os.cpu_count() or 4)

# ImageNet normalization (as before)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# =========================
# 4) UTILITIES
# =========================

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")

def read_split_list(path):
    with open(path, "r") as f:
        ids = [ln.strip() for ln in f if ln.strip() and not ln.strip().startswith("#")]
    return ids

def resolve_image_path(identifier):
    """
    identifier can be:
      - a full/relative path to an image, OR
      - a bare stem (ID) without extension.
    Tries to find the image file under IMAGE_DIR_CANDIDATES.
    """
    s = identifier.replace("\\", "/")
    base, ext = os.path.splitext(s)

    # Case 1: identifier already has an extension (.jpg/.png/...)
    if ext.lower() in IMAGE_EXTS:
        # try as-is
        if os.path.exists(s):
            return s
        # try relative to each candidate dir
        for d in IMAGE_DIR_CANDIDATES:
            cand = os.path.join(d, s)
            if os.path.exists(cand):
                return cand
        # try relative to the splits dir root
        cand = os.path.join(os.path.dirname(SPLITS_DIR), s)
        return cand  # may not exist yet but we tried

    # Case 2: bare ID (no extension)
    stem = os.path.basename(base)
    for d in IMAGE_DIR_CANDIDATES:
        for e in IMAGE_EXTS:
            cand = os.path.join(d, stem + e)
            if os.path.exists(cand):
                return cand
    # Fallback: first candidate with .jpg
    return os.path.join(IMAGE_DIR_CANDIDATES[0], stem + ".jpg")

def load_class_dict(csv_path):
    """Return (class_names, color2idx, ignore_index, unlabeled_index)."""
    df = pd.read_csv(csv_path, skipinitialspace=True)
    # normalize headers
    df.columns = [c.strip().lower() for c in df.columns]
    assert set(["name","r","g","b"]).issubset(df.columns), "class_dict.csv must have columns: name,r,g,b"

    names = [str(x).strip() for x in df["name"].tolist()]
    colors = [(int(r), int(g), int(b)) for r, g, b in zip(df["r"], df["g"], df["b"])]
    color2idx = {tuple(rgb): idx for idx, rgb in enumerate(colors)}

    ignore_index = names.index("conflicting") if "conflicting" in names else None
    unlabeled_index = names.index("unlabeled") if "unlabeled" in names else 0
    return names, color2idx, ignore_index, unlabeled_index

def rgb_mask_to_index(mask_rgb, color2idx, default_idx=0):
    """
    Convert an RGB mask (H,W,3) into a class-index mask (H,W) using color2idx.
    Any pixel color not present in color2idx maps to default_idx (usually 'unlabeled').
    """
    h, w = mask_rgb.shape[:2]
    out = np.full((h, w), default_idx, dtype=np.int64)
    # loop over known colors (only ~24), boolean-assign each — fast enough and robust
    for (r, g, b), cls in color2idx.items():
        matches = (mask_rgb[..., 0] == r) & (mask_rgb[..., 1] == g) & (mask_rgb[..., 2] == b)
        out[matches] = cls
    return out

def build_items_from_split(split_txt, masks_dir):
    """
    Returns list of dicts: {id, img, mask}, skipping samples with missing files.
    """
    ids = read_split_list(split_txt)
    items, missing = [], []
    for ident in ids:
        img_path = resolve_image_path(ident)
        stem = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(masks_dir, stem + ".png")
        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            missing.append((img_path, mask_path))
            continue
        items.append({"id": stem, "img": img_path, "mask": mask_path})
    if missing:
        print(f"[warn] {len(missing)} samples missing image or mask — they were skipped. Examples:", missing[:3])
    return items

In [ ]:
# =========================
# 5) DATASET
# =========================
class SemanticDatasetFromSplits(Dataset):
    def __init__(self, items, color2idx, mean, std, transform=None):
        self.items = items
        self.color2idx = color2idx
        self.transform = transform
        self.img_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        rec = self.items[idx]

        # image
        img = cv2.imread(rec["img"], cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f"Cannot read image: {rec['img']}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # mask (RGB color -> class index)
        mask_rgb = np.array(Image.open(rec["mask"]).convert("RGB"))
        mask_idx = rgb_mask_to_index(mask_rgb, self.color2idx, default_idx=0)

        if self.transform is not None:
            aug = self.transform(image=img, mask=mask_idx)
            img = aug["image"]
            mask_idx = aug["mask"]

        img = Image.fromarray(img)
        img = self.img_tf(img)
        mask = torch.from_numpy(mask_idx).long()
        return img, mask


In [ ]:
# =========================
# 6) TRANSFORMS
# =========================
t_train = A.Compose([
    A.Resize(IMG_H, IMG_W, interpolation=cv2.INTER_LINEAR,  # images bilinear
             mask_interpolation=cv2.INTER_NEAREST, always_apply=True),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.GridDistortion(p=0.2),
    A.RandomBrightnessContrast((0, 0.5), (0, 0.5), p=0.5),
    A.GaussNoise(p=0.2),
])

t_val = A.Compose([
    A.Resize(IMG_H, IMG_W, interpolation=cv2.INTER_LINEAR,
             mask_interpolation=cv2.INTER_NEAREST, always_apply=True),
])


In [ ]:
# =========================
# 7) METRICS (ignore 'conflicting')
# =========================
def pixel_accuracy(output, target, ignore_index=None):
    with torch.no_grad():
        pred = torch.argmax(output, dim=1)
        if ignore_index is not None:
            valid = target != ignore_index
            correct = ((pred == target) & valid).sum().float()
            total = valid.sum().float().clamp_min(1.0)
        else:
            correct = (pred == target).sum().float()
            total = torch.tensor(target.numel(), device=output.device, dtype=torch.float32)
        return (correct / total).item()

def mIoU(output, target, n_classes, ignore_index=None, smooth=1e-10):
    with torch.no_grad():
        pred = torch.argmax(output, dim=1)
        ious = []
        for cls in range(n_classes):
            if ignore_index is not None and cls == ignore_index:
                continue
            pred_i = (pred == cls)
            target_i = (target == cls)
            if ignore_index is not None:
                valid = (target != ignore_index)
                pred_i = pred_i & valid
                target_i = target_i & valid
            inter = torch.logical_and(pred_i, target_i).sum().float()
            union = torch.logical_or(pred_i, target_i).sum().float()
            if target_i.sum().item() == 0:
                ious.append(np.nan)  # class not in GT for this batch
            else:
                ious.append(((inter + smooth) / (union + smooth)).item())
        return np.nanmean(ious)

def get_lr(optimizer):
    return optimizer.param_groups[0]["lr"]


In [ ]:
# =========================
# 8) TRAIN / EVAL LOOP
# =========================
def fit(epochs, model, train_loader, val_loader, criterion, optimizer, scheduler,
        n_classes, ignore_index=None, device=DEVICE, ckpt_dir=OUTPUT_DIR, ckpt_name_prefix="Unet-Mobilenet_v2"):
    torch.cuda.empty_cache()
    best_val = float("inf")
    patience = 7
    bad_epochs = 0

    history = {
        "train_loss": [], "val_loss": [],
        "train_miou": [], "val_miou": [],
        "train_acc": [], "val_acc": [],
        "lrs": []
    }

    model.to(device)
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        since = time.time()
        model.train()

        train_loss = 0.0
        train_miou, train_acc = 0.0, 0.0

        for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [train]"):
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            scheduler.step()

            # metrics
            train_loss += loss.item()
            train_miou += mIoU(outputs, masks, n_classes, ignore_index)
            train_acc  += pixel_accuracy(outputs, masks, ignore_index)
            history["lrs"].append(get_lr(optimizer))

        # averages
        train_loss /= max(1, len(train_loader))
        train_miou /= max(1, len(train_loader))
        train_acc  /= max(1, len(train_loader))

        # validation
        model.eval()
        val_loss = 0.0
        val_miou, val_acc = 0.0, 0.0
        with torch.no_grad():
            for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [val]"):
                imgs = imgs.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True)
                outputs = model(imgs)
                loss = criterion(outputs, masks)
                val_loss += loss.item()
                val_miou += mIoU(outputs, masks, n_classes, ignore_index)
                val_acc  += pixel_accuracy(outputs, masks, ignore_index)

        val_loss /= max(1, len(val_loader))
        val_miou /= max(1, len(val_loader))
        val_acc  /= max(1, len(val_loader))

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_miou"].append(train_miou)
        history["val_miou"].append(val_miou)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch:02d}/{epochs} | "
              f"Train: loss {train_loss:.4f}, mIoU {train_miou:.4f}, acc {train_acc:.4f} | "
              f"Val: loss {val_loss:.4f}, mIoU {val_miou:.4f}, acc {val_acc:.4f} | "
              f"LR {get_lr(optimizer):.3e} | "
              f"Time {(time.time()-since)/60:.2f}m")

        # checkpointing + early stopping
        if val_loss < best_val:
            best_val = val_loss
            bad_epochs = 0
            ckpt_path = os.path.join(ckpt_dir, f"{ckpt_name_prefix}_best.pth")
            torch.save(model, ckpt_path)
            print(f"  ↳ Saved best model to: {ckpt_path}")
        else:
            bad_epochs += 1
            print(f"  ↳ No improvement ({bad_epochs}/{patience})")
            if bad_epochs >= patience:
                print("Early stopping.")
                break

    print(f"Total training time: {(time.time() - start_time)/60:.2f} m")
    return history

In [ ]:
# =========================
# 9) BUILD DATASETS/LOADERS FROM YOUR SPLITS
# =========================
class_names, color2idx, IGNORE_IDX, UNLABELED_IDX = load_class_dict(CLASS_CSV)
N_CLASSES = len(class_names)

print("Classes:", class_names)
print("Ignore index:", IGNORE_IDX, "(conflicting)" if IGNORE_IDX is not None else "")

split_paths = {k: os.path.join(SPLITS_DIR, v) for k, v in SPLIT_FILENAMES.items()}
for k, p in split_paths.items():
    if not os.path.exists(p):
        print(f"[warn] Split file missing: {p}")

train_items = build_items_from_split(split_paths["train"], MASKS_DIR)
val_items   = build_items_from_split(split_paths["val"],   MASKS_DIR)
# Optional: test set for later evaluation
test_items  = build_items_from_split(split_paths["test"],  MASKS_DIR) if os.path.exists(split_paths["test"]) else []

train_set = SemanticDatasetFromSplits(train_items, color2idx, MEAN, STD, t_train)
val_set   = SemanticDatasetFromSplits(val_items,   color2idx, MEAN, STD, t_val)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)

print(f"Train samples: {len(train_set)} | Val samples: {len(val_set)} | Test (optional): {len(test_items)}")


In [ ]:
# =========================
# 10) MODEL, LOSS, OPTIM, SCHED
# =========================
model = smp.Unet(
    encoder_name="mobilenet_v2",
    encoder_weights="imagenet",
    classes=N_CLASSES,           # includes 'conflicting'; we ignore it in the loss
    activation=None,
    encoder_depth=5,
    decoder_channels=[256, 128, 64, 32, 16]
)

criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_IDX) if IGNORE_IDX is not None else nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=MAX_LR, epochs=EPOCHS,
                                            steps_per_epoch=len(train_loader))

In [ ]:
# =========================
# 11) TRAIN
# =========================
history = fit(
    epochs=EPOCHS, model=model,
    train_loader=train_loader, val_loader=val_loader,
    criterion=criterion, optimizer=optimizer, scheduler=sched,
    n_classes=N_CLASSES, ignore_index=IGNORE_IDX, device=DEVICE,
    ckpt_dir=OUTPUT_DIR, ckpt_name_prefix="Unet-Mobilenet_v2"
)

# Save final model as well
final_path = os.path.join(OUTPUT_DIR, "Unet-Mobilenet_v2_last.pth")
torch.save(model, final_path)
print("Saved final model to:", final_path)

## Evaluation